# 🌵 Drought Module — Steps 1–7
### Climate Risk Scoring System for Indian Real Estate

This notebook extends the single-city Jodhpur analysis to:
1. Fetch data for 10 diverse Indian cities
2. Compute SPI (3-month and 12-month)
3. Compute max dry spell per city per year
4. Build a yearly training dataframe
5. Train an XGBoost drought classifier
6. Generate a 0–100 drought risk score
7. Save all outputs to `data/outputs/`

---
## Step 1 — Multi-city Data Fetching
Fetch daily rainfall and ET0 from Open-Meteo for 10 Indian cities (2015–2023).

In [ ]:
import requests
import pandas as pd
import numpy as np
import time

# ----- City registry -----
# Each city has a name, latitude, longitude, and a qualitative drought profile
# for reference (used only as context — not as a feature).
CITIES = [
    {"name": "Jodhpur",    "lat": 26.24, "lon": 73.02},   # Rajasthan desert
    {"name": "Jaisalmer",  "lat": 26.92, "lon": 70.91},   # Extreme drought
    {"name": "Bikaner",    "lat": 28.02, "lon": 73.31},   # Arid
    {"name": "Nagpur",     "lat": 21.15, "lon": 79.09},   # Moderate
    {"name": "Pune",       "lat": 18.52, "lon": 73.86},   # Moderate
    {"name": "Chennai",    "lat": 13.08, "lon": 80.27},   # Coastal / periodic drought
    {"name": "Bengaluru",  "lat": 12.97, "lon": 77.59},   # Moderate
    {"name": "Hyderabad",  "lat": 17.39, "lon": 78.49},   # Moderate
    {"name": "Bhopal",     "lat": 23.26, "lon": 77.41},   # Central India
    {"name": "Patna",      "lat": 25.59, "lon": 85.14},   # North / wetter
]

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"
START    = "2015-01-01"
END      = "2023-12-31"

def fetch_city(city: dict) -> pd.DataFrame:
    """Fetch daily precipitation and ET0 for one city."""
    params = {
        "latitude":   city["lat"],
        "longitude":  city["lon"],
        "start_date": START,
        "end_date":   END,
        "daily":      ["precipitation_sum", "et0_fao_evapotranspiration"],
        "timezone":   "Asia/Kolkata",
    }
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()            # crash loudly on HTTP errors
    data = response.json()["daily"]

    df = pd.DataFrame(data)
    df.columns = ["date", "rainfall_mm", "et0_mm"]
    df["date"] = pd.to_datetime(df["date"])

    # Tag every row with city metadata
    df["city"] = city["name"]
    df["lat"]  = city["lat"]
    df["lon"]  = city["lon"]
    return df

# ----- Fetch all cities -----
all_dfs = []
for city in CITIES:
    print(f"  Fetching {city['name']} ...", end=" ")
    df_city = fetch_city(city)
    all_dfs.append(df_city)
    print(f"✓  ({len(df_city)} rows)")
    time.sleep(0.3)    # polite pause so we don't hammer the API

# ----- Combine into master dataframe -----
df_all = pd.concat(all_dfs, ignore_index=True)

# Fill any missing values with 0 (rare for rainfall; ET0 gaps are unusual)
df_all[["rainfall_mm", "et0_mm"]] = df_all[["rainfall_mm", "et0_mm"]].fillna(0)

print(f"\nMaster dataframe shape: {df_all.shape}")
print(df_all.head(5))

---
## Step 2 — Compute SPI (Standardised Precipitation Index)

**SPI** standardises accumulated rainfall using a Gamma distribution so that all
locations share the same scale. Values below −1.0 indicate drought; below −2.0
indicate severe drought.

**Formula used:**
1. Compute a rolling precipitation sum (`window` = 90 days for SPI-3, 365 days for SPI-12).
2. Fit a Gamma distribution to the *historical* (all-year) rolling sums for each city.
3. Correct for the probability of exactly-zero rainfall: $H(x) = q + (1-q)\cdot G(x)$.
4. Map the cumulative probability through the inverse standard-normal: $\text{SPI} = \Phi^{-1}(H(x))$.

In [ ]:
from scipy.stats import gamma as gamma_dist, norm as norm_dist
import warnings

def compute_spi(series: pd.Series, window: int) -> pd.Series:
    """
    Compute the Standardised Precipitation Index (SPI) for a daily rainfall
    series using a rolling accumulation window and Gamma distribution fitting.

    Parameters
    ----------
    series : pd.Series
        Daily rainfall (mm), already sorted by date.
    window : int
        Number of days to accumulate (e.g. 90 for SPI-3, 365 for SPI-12).

    Returns
    -------
    pd.Series  –  SPI values aligned to the original index.
    """
    # 1. Rolling accumulation
    rolled = series.rolling(window=window, min_periods=window).sum()

    # 2. Use all non-NaN accumulated values to fit the Gamma distribution
    valid = rolled.dropna().values

    if len(valid) < 10:
        # Not enough data to fit — return all NaN
        return pd.Series(np.nan, index=series.index)

    # 3. Probability of zero (or near-zero) rainfall
    eps = 1e-6
    q = np.mean(valid <= eps)          # P(x == 0)
    non_zero = valid[valid > eps]

    if len(non_zero) < 5:
        return pd.Series(np.nan, index=series.index)

    # 4. Fit Gamma to non-zero values
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha, loc, beta = gamma_dist.fit(non_zero, floc=0)

    # 5. Compute mixed CDF H(x) for every accumulated value
    def mixed_cdf(x):
        if x <= eps:
            return q
        return q + (1 - q) * gamma_dist.cdf(x, a=alpha, loc=loc, scale=beta)

    H = rolled.apply(lambda x: mixed_cdf(x) if not np.isnan(x) else np.nan)

    # 6. Clip to avoid Inf from ppf(0) or ppf(1)
    H_clipped = H.clip(lower=1e-6, upper=1 - 1e-6)

    # 7. Inverse normal CDF → SPI
    spi = norm_dist.ppf(H_clipped)
    spi[H.isna()] = np.nan

    return pd.Series(spi, index=series.index)


# ----- Apply per city (SPI must be computed on each city independently) -----
spi3_list  = []
spi12_list = []

for city_name, grp in df_all.groupby("city"):
    grp = grp.sort_values("date")
    spi3  = compute_spi(grp["rainfall_mm"], window=90)
    spi12 = compute_spi(grp["rainfall_mm"], window=365)
    spi3_list.append(spi3)
    spi12_list.append(spi12)

df_all = df_all.sort_values(["city", "date"]).reset_index(drop=True)
df_all["spi_3month"]  = pd.concat(spi3_list).values
df_all["spi_12month"] = pd.concat(spi12_list).values

print("SPI columns added. Sample (Jodhpur):")
print(df_all[df_all["city"] == "Jodhpur"][["date", "rainfall_mm", "spi_3month", "spi_12month"]].dropna().head(10))

---
## Step 3 — Compute Max Dry Spell per City per Year

A **dry spell** is a consecutive run of days with rainfall < 1 mm.
`max_dry_spell_days` = longest such run in a given calendar year.

In [ ]:
def max_dry_spell(is_dry: pd.Series) -> int:
    """
    Find the longest consecutive run of True values in a boolean Series.
    Uses a cumsum trick: every time is_dry flips to False, the group counter
    increments, isolating each dry/wet run.
    """
    if is_dry.sum() == 0:
        return 0
    # Group runs of consecutive dry days; sum each group's length
    groups = is_dry.groupby((~is_dry).cumsum())
    return int(groups.sum().max())

# Add helper columns
df_all["is_dry_day"]      = df_all["rainfall_mm"] < 1.0
df_all["year"]            = df_all["date"].dt.year
df_all["water_deficit_mm"] = df_all["et0_mm"] - df_all["rainfall_mm"]

# Compute max dry spell for each (city, year) pair
max_dry = (
    df_all.groupby(["city", "year"])["is_dry_day"]
    .apply(max_dry_spell)
    .reset_index()
    .rename(columns={"is_dry_day": "max_dry_spell_days"})
)

print("Max dry spell (days) per city per year:")
print(max_dry.pivot(index="year", columns="city", values="max_dry_spell_days").to_string())

---
## Step 4 — Build the Yearly Training Dataframe

Aggregate daily observations into one row per `(city, year)`.  
**Target variable:** `drought_label = 1` if `spi_12month_mean < −1.0`, else 0.

> ⚠️ `spi_12month_mean` is used **only** to create the label; it is **excluded** from
> the feature matrix to prevent data leakage.

In [ ]:
# ----- Yearly aggregation -----
yearly = (
    df_all.groupby(["city", "year"])
    .agg(
        lat             = ("lat",              "first"),
        lon             = ("lon",              "first"),
        total_rainfall  = ("rainfall_mm",      "sum"),
        total_et0       = ("et0_mm",           "sum"),
        total_deficit   = ("water_deficit_mm", "sum"),
        dry_days        = ("is_dry_day",       "sum"),
        spi_3month_mean = ("spi_3month",       "mean"),
        spi_12month_mean= ("spi_12month",      "mean"),
    )
    .reset_index()
)

# ----- Merge max dry spell -----
yearly = yearly.merge(max_dry, on=["city", "year"], how="left")

# ----- Drought label (from SPI-12) -----
yearly["drought_label"] = (yearly["spi_12month_mean"] < -1.0).astype(int)

# ----- Column order as specified in brief -----
col_order = [
    "city", "year", "lat", "lon",
    "total_rainfall", "total_et0", "total_deficit",
    "dry_days", "max_dry_spell_days",
    "spi_3month_mean", "spi_12month_mean",
    "drought_label",
]
yearly = yearly[col_order]

print(f"Yearly dataframe shape: {yearly.shape}")
print(f"Drought years (label=1): {yearly['drought_label'].sum()} / {len(yearly)}")
print()
print(yearly.head(20).to_string(index=False))

---
## Step 5 — Train a Drought Prediction Model

- **Train:** years 2015–2021
- **Test:** years 2022–2023
- **Model:** XGBoost classifier
- **Features:** all columns except `city`, `year`, `lat`, `lon`, `spi_12month_mean`, `drought_label`
- **Metrics:** Accuracy, AUC-ROC, Precision, Recall, F1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    classification_report,
)

# ----- Feature / target definition -----
# NOTE: spi_12month_mean is the BASIS for the label → excluded to avoid leakage
FEATURE_COLS = [
    "total_rainfall",
    "total_et0",
    "total_deficit",
    "dry_days",
    "max_dry_spell_days",
    "spi_3month_mean",
]
TARGET_COL = "drought_label"

# Drop rows where SPI is NaN (first ~year of data per city due to rolling window)
yearly_clean = yearly.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()

# ----- Train / test split by year -----
train = yearly_clean[yearly_clean["year"] <= 2021]
test  = yearly_clean[yearly_clean["year"] >= 2022]

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

print(f"Train rows: {len(train)}  |  Test rows: {len(test)}")
print(f"Train positive rate: {y_train.mean():.2%}")
print(f"Test  positive rate: {y_test.mean():.2%}")

# ----- XGBoost model -----
model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
)
model.fit(X_train, y_train)

# ----- Evaluation -----
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)

# AUC-ROC — handle edge case where only one class is present in test set
try:
    auc = roc_auc_score(y_test, y_pred_prob)
except ValueError:
    auc = float("nan")

print(f"\n{'='*40}")
print(f"  Accuracy  : {acc:.4f}")
print(f"  AUC-ROC   : {auc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"{'='*40}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# ----- Feature importance plot -----
importances = model.feature_importances_
feat_df = pd.DataFrame({"feature": FEATURE_COLS, "importance": importances})
feat_df = feat_df.sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feat_df["feature"], feat_df["importance"], color="steelblue")
ax.set_xlabel("XGBoost Feature Importance (weight)")
ax.set_title("Drought Model — Feature Importances")
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.savefig("../data/outputs/feature_importance.png", dpi=150)
plt.show()
print("Feature importance chart saved to data/outputs/feature_importance.png")

---
## Step 6 — Generate Drought Risk Score (0–100)

Convert the model's probability output to a 0–100 score and assign a risk category:

| Score | Category  |
|-------|-----------|
| 0–25  | LOW       |
| 26–50 | MEDIUM    |
| 51–75 | HIGH      |
| 76–100| VERY HIGH |

In [ ]:
def prob_to_score(p: float) -> float:
    """Convert probability [0, 1] to a 0–100 drought risk score."""
    return round(p * 100, 2)

def score_to_category(score: float) -> str:
    """Assign a categorical risk level from a 0–100 score."""
    if score <= 25:
        return "LOW"
    elif score <= 50:
        return "MEDIUM"
    elif score <= 75:
        return "HIGH"
    else:
        return "VERY HIGH"

# ----- Score the entire dataset (train + test) -----
all_probs  = model.predict_proba(yearly_clean[FEATURE_COLS])[:, 1]
risk_scores = yearly_clean[["city", "year", "lat", "lon", "drought_label"]].copy()
risk_scores["drought_prob"]     = all_probs.round(4)
risk_scores["drought_risk_score"] = risk_scores["drought_prob"].apply(prob_to_score)
risk_scores["risk_category"]    = risk_scores["drought_risk_score"].apply(score_to_category)

# ----- Summary -----
print("Drought Risk Score Summary (all cities, all years):")
print(risk_scores.to_string(index=False))

# ----- Highlight worst years per city -----
worst = (
    risk_scores.groupby("city")
    .apply(lambda g: g.loc[g["drought_risk_score"].idxmax()])
    .reset_index(drop=True)[["city", "year", "drought_risk_score", "risk_category"]]
    .sort_values("drought_risk_score", ascending=False)
)
print("\nWorst drought year per city:")
print(worst.to_string(index=False))

---
## Step 7 — Save Outputs

| File | Description |
|------|-------------|
| `data/outputs/drought_features.csv` | Yearly training dataframe |
| `data/outputs/drought_model.pkl` | Trained XGBoost model |
| `data/outputs/drought_risk_scores.csv` | Risk score summary table |

In [ ]:
import pickle
import os

# ----- Ensure output directory exists -----
OUTPUT_DIR = "../data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Yearly features CSV
features_path = os.path.join(OUTPUT_DIR, "drought_features.csv")
yearly.to_csv(features_path, index=False)
print(f"✓  Saved yearly features  → {features_path}")

# 2. Trained model (pickle)
model_path = os.path.join(OUTPUT_DIR, "drought_model.pkl")
with open(model_path, "wb") as f:
    pickle.dump(model, f)
print(f"✓  Saved model            → {model_path}")

# 3. Risk score summary CSV
scores_path = os.path.join(OUTPUT_DIR, "drought_risk_scores.csv")
risk_scores.to_csv(scores_path, index=False)
print(f"✓  Saved risk scores      → {scores_path}")

print("\nAll outputs saved successfully!")